# Swedish grammaticality scorer: full reproduction on Colab
Runtime → Change runtime type → **T4 GPU**. Set `HF_USER` below. Total run time is about 30–40 min.

If the install cell asks you to **restart the session**, do it, then run the cells again from the top (the clone step will say the folder exists — that's fine).

In [ ]:
HF_USER = "Nawazaz"  # your Hugging Face username
GITHUB_REPO = "https://github.com/Nawazaz/swedish-grammar-scorer"
DATASET_ID = f"{HF_USER}/swedish-grammar-data"
MODEL_ID = f"{HF_USER}/swedish-grammar-scorer"

In [ ]:
!git clone -q $GITHUB_REPO repo
%cd repo
!pip install -q -r requirements.txt
# Colab's preinstalled torchvision/torchaudio don't match the pinned torch and break transformers
!pip uninstall -q -y torchvision torchaudio
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

Log in with a Hugging Face token that has **write** access (huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Build and publish the dataset (~5 min)

In [ ]:
!python build_dataset.py --out data/processed --push $DATASET_ID

## 2. Fine-tune and publish the model (~20 min on a T4)

In [ ]:
!python train.py --data data/processed --output-dir models/scorer --push $MODEL_ID

## 3. Evaluate exactly like the organisers (loads the model from the Hub)

In [ ]:
!python evaluate.py --model $MODEL_ID

Optional: CPU speed, and ablations for the paper.

In [ ]:
!CUDA_VISIBLE_DEVICES= python evaluate.py --model $MODEL_ID --output scores_cpu.jsonl

In [ ]:
# Ablation examples (local only, no push):
# !python build_dataset.py --out data/no_wiki --no-wikipedia
# !python train.py --data data/no_wiki --output-dir models/no_wiki
# !python predict.py --model models/no_wiki --input data/starter/test.jsonl --output s.jsonl